# LeetCode 583 — Delete Operation for Two Strings bằng Python

Link đề: https://leetcode.com/problems/delete-operation-for-two-strings/ · Độ khó: **Medium** · Chủ đề: String, Dynamic Programming (quy hoạch động), Longest Common Subsequence (dãy con chung dài nhất)

Skeleton LeetCode:

```python
class Solution:
    def minDistance(self, word1: str, word2: str) -> int:
        ...
```

Ghi chú: bài này là "anh em" của Edit Distance (72) nhưng **chỉ được xóa**, không chèn/thay. Điều đó làm bài quy về một khái niệm rất đẹp: giữ lại **dãy con chung dài nhất** rồi xóa mọi thứ còn lại. Notebook đi từ đệ quy vét cạn → ghi nhớ → bảng DP → công thức LCS → LCS bằng bisect (kiểu LIS).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi bước xóa **đúng một ký tự** ở **một trong hai** chuỗi; cần ít bước nhất để hai chuỗi **giống hệt nhau**.
2. Nắm các công cụ Python cần thiết: cắt chuỗi `s[:i]` (tiền tố), chỉ số, `enumerate(..., 1)`, tạo bảng 2 chiều **đúng cách**, `functools.lru_cache`, `bisect_left`.
3. Phân biệt **substring** (chuỗi con liền mạch) với **subsequence** (dãy con, được bỏ ký tự nhưng giữ thứ tự), và thấy rằng `đáp án = len(word1) + len(word2) - 2 * LCS`.
4. Đi qua 5 cách: đệ quy thuần (chậm), đệ quy có nhớ, bảng DP 2 chiều, LCS với mảng 1 chiều (tiết kiệm bộ nhớ), LCS bằng danh sách vị trí + `bisect` (đưa về LIS).
5. Tránh các lỗi kinh điển: khởi tạo biên bằng 0, nhầm với Edit Distance (`f[i-1][j-1] + 1`), quên nhân đôi LCS, `[[0]*n]*m` (các hàng dùng chung), đếm ký tự chung mà bỏ qua thứ tự.

## 2. Đề bài & ví dụ

Cho hai chuỗi `word1` và `word2`. Trả về **số bước tối thiểu** để làm cho `word1` và `word2` **giống nhau**. Trong mỗi bước bạn được xóa **đúng một ký tự** ở **một trong hai chuỗi**.

**Ví dụ 1**

- Input: `word1 = "sea"`, `word2 = "eat"`
- Output: `2`
- Giải thích: xóa `'s'` khỏi `"sea"` được `"ea"`, xóa `'t'` khỏi `"eat"` được `"ea"`. Hai bước.

**Ví dụ 2**

- Input: `word1 = "leetcode"`, `word2 = "etco"`
- Output: `4`
- Giải thích: `"etco"` là dãy con của `"leetcode"` nên chỉ cần xóa 4 ký tự `l, e, d, e` ở `"leetcode"`.

**Ràng buộc:**

- `1 <= word1.length, word2.length <= 500`
- `word1` và `word2` chỉ gồm chữ cái tiếng Anh thường.

Lưu ý: kết quả sau khi xóa có thể là **chuỗi rỗng** (ví dụ `"a"` và `"b"` -> xóa cả hai, 2 bước). Hai chuỗi rỗng "giống nhau", nên luôn luôn có đáp án (tối đa `m + n`).

In [1]:
from typing import List
from functools import lru_cache
from itertools import combinations
from bisect import bisect_left
from collections import defaultdict, Counter
import random, time, sys

examples = [
    ("sea", "eat", 2),
    ("leetcode", "etco", 4),
]
for w1, w2, expected in examples:
    print(f"word1={w1!r:12} word2={w2!r:8} len=({len(w1)},{len(w2)}) kỳ vọng={expected}")

word1='sea'        word2='eat'    len=(3,3) kỳ vọng=2
word1='leetcode'   word2='etco'   len=(8,4) kỳ vọng=4


## 3. Kiến thức nền

### 3.1. Chỉ số, cắt chuỗi và "tiền tố"

Chuỗi Python đánh chỉ số từ `0`. `s[:i]` là **tiền tố** (prefix) gồm `i` ký tự đầu, `s[i]` là ký tự thứ `i + 1`. Trong DP chuỗi ta hay nói "`i` ký tự đầu" và ký tự cuối của nó là `s[i - 1]`. Đây là chỗ người mới hay lệch một đơn vị.

In [2]:
s = "leetcode"
for i in range(0, len(s) + 1):
    last = s[i - 1] if i > 0 else "(rỗng)"
    print(f"i={i}: s[:{i}] = {s[:i]!r:11} ký tự cuối của tiền tố = {last}")

i=0: s[:0] = ''          ký tự cuối của tiền tố = (rỗng)
i=1: s[:1] = 'l'         ký tự cuối của tiền tố = l
i=2: s[:2] = 'le'        ký tự cuối của tiền tố = e
i=3: s[:3] = 'lee'       ký tự cuối của tiền tố = e
i=4: s[:4] = 'leet'      ký tự cuối của tiền tố = t
i=5: s[:5] = 'leetc'     ký tự cuối của tiền tố = c
i=6: s[:6] = 'leetco'    ký tự cuối của tiền tố = o
i=7: s[:7] = 'leetcod'   ký tự cuối của tiền tố = d
i=8: s[:8] = 'leetcode'  ký tự cuối của tiền tố = e


### 3.2. Substring và subsequence

- **Substring** (chuỗi con): các ký tự **liền nhau**. `"eet"` là substring của `"leetcode"`.
- **Subsequence** (dãy con): được **bỏ** một số ký tự (kể cả không bỏ hoặc bỏ hết) nhưng **giữ nguyên thứ tự**. `"etco"` là dãy con của `"leetcode"` (e-t-c-o), dù không liền nhau.

Một chuỗi độ dài `m` có tối đa `2^m` dãy con (mỗi ký tự: giữ hoặc bỏ). Xóa ký tự ở bài này **chính là** chọn một dãy con để giữ lại.

In [3]:
def is_subsequence(small: str, big: str) -> bool:
    it = iter(big)
    return all(ch in it for ch in small)   # mỗi ch phải tìm thấy phía sau ký tự trước

print(is_subsequence("etco", "leetcode"))   # True
print(is_subsequence("eet", "leetcode"))    # True
print(is_subsequence("tee", "leetcode"))    # False (sai thứ tự)
print("eet" in "leetcode", "etco" in "leetcode")   # substring: True False

def all_subsequences(s: str):
    n = len(s)
    out = set()
    for mask in range(1 << n):
        out.add("".join(s[k] for k in range(n) if mask >> k & 1))
    return out

print(sorted(all_subsequences("abc")))   # 2^3 = 8 dãy con

True
True
False
True True
['', 'a', 'ab', 'abc', 'ac', 'b', 'bc', 'c']


### 3.3. Bảng 2 chiều: tạo đúng cách

DP chuỗi cần bảng `f` kích thước `(m + 1) x (n + 1)`. Cách đúng là list comprehension (mỗi hàng là một list **mới**). Cách sai `[[0] * (n + 1)] * (m + 1)` chỉ nhân bản **tham chiếu tới cùng một hàng**, sửa một ô thì cả cột đổi theo.

In [4]:
m, n = 3, 4
good = [[0] * (n + 1) for _ in range(m + 1)]
bad = [[0] * (n + 1)] * (m + 1)
good[1][2] = 7
bad[1][2] = 7
print("đúng:", good)
print("sai :", bad)
print("các hàng của bad là cùng một đối tượng?", bad[0] is bad[1] is bad[2])

đúng: [[0, 0, 0, 0, 0], [0, 0, 7, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
sai : [[0, 0, 7, 0, 0], [0, 0, 7, 0, 0], [0, 0, 7, 0, 0], [0, 0, 7, 0, 0]]
các hàng của bad là cùng một đối tượng? True


### 3.4. `enumerate(s, 1)`, `min`, và `lru_cache`

- `enumerate(s, 1)` cho `(1, s[0]), (2, s[1]), ...` — chỉ số bắt đầu từ 1, rất hợp khi bảng có thêm hàng/cột 0 làm "biên".
- `min(a, b)` lấy số nhỏ hơn.
- `@lru_cache(None)` bọc một hàm đệ quy để **nhớ kết quả** theo đối số. Gọi lại với cùng đối số thì trả ngay, không tính lại. Đối số phải "băm được" (số, chuỗi, tuple).

In [5]:
for i, ch in enumerate("sea", 1):
    print(i, ch)

calls = 0
@lru_cache(None)
def fib(k):
    global calls
    calls += 1
    return k if k < 2 else fib(k - 1) + fib(k - 2)

print(fib(30), "số lần thân hàm thực sự chạy:", calls)   # 31, không phải ~2 triệu

1 s
2 e
3 a
832040 số lần thân hàm thực sự chạy: 31


### 3.5. `bisect_left` (dùng ở Cách 5)

`bisect_left(a, x)` trả về vị trí **đầu tiên** trong list đã sắp xếp `a` mà `a[pos] >= x` (tìm kiếm nhị phân, `O(log n)`). Nó cho biết "x nên chèn vào đâu để list vẫn tăng".

In [6]:
a = [2, 5, 5, 9]
print(bisect_left(a, 5), bisect_left(a, 6), bisect_left(a, 1), bisect_left(a, 100))

1 3 0 4


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Xét `"sea"` và `"eat"`. Sau khi xóa xong, hai chuỗi phải bằng nhau; gọi chuỗi cuối cùng là `T`. Khi đó:

- `T` là **dãy con của `word1`** (vì ta chỉ xóa, không đổi thứ tự).
- `T` là **dãy con của `word2`**.
- Số bước = `(m - len(T)) + (n - len(T))`.

Muốn **ít bước nhất** thì `len(T)` phải **lớn nhất**. Vậy `T` chính là **dãy con chung dài nhất** (LCS - Longest Common Subsequence).

Tính tay với `sea` / `eat`: các dãy con chung là `""`, `"e"`, `"a"`, `"ea"`. Dài nhất là `"ea"` (độ dài 2). Đáp án = `(3 - 2) + (3 - 2) = 2`. Khớp ví dụ.

In [7]:
def common_subsequences(a: str, b: str):
    return sorted(all_subsequences(a) & all_subsequences(b), key=lambda x: (len(x), x))

cs = common_subsequences("sea", "eat")
print("dãy con chung:", cs)
best = max(cs, key=len)
print("dài nhất:", repr(best), "-> bước =", (3 - len(best)) + (3 - len(best)))

cs2 = common_subsequences("leetcode", "etco")
best2 = max(cs2, key=len)
print("leetcode / etco: dài nhất", repr(best2), "-> bước =", (8 - len(best2)) + (4 - len(best2)))

dãy con chung: ['', 'a', 'e', 'ea']
dài nhất: 'ea' -> bước = 2
leetcode / etco: dài nhất 'etco' -> bước = 4


Đã có công thức nhưng **làm sao tìm LCS** nhanh? Thử mọi dãy con của `word1` (2^500) là không thể. Ta cần nhìn bài toán theo từng ký tự cuối: đó là cách đi vào đệ quy.

## 5. Cách 1 — Brute Force: đệ quy thuần theo hai con trỏ

**Ý tưởng.** Gọi `dfs(i, j)` = số bước tối thiểu để làm cho `word1[i:]` và `word2[j:]` giống nhau (hai đuôi). Xét hai ký tự đầu của hai đuôi:

- Nếu `word1[i] == word2[j]`: giữ cả hai, bài toán còn `dfs(i + 1, j + 1)`. (Giữ chúng không bao giờ tệ hơn: sẽ chứng minh ở mục 10.)
- Nếu khác nhau: **ít nhất một** trong hai phải bị xóa. Xóa `word1[i]` -> `1 + dfs(i + 1, j)`; hoặc xóa `word2[j]` -> `1 + dfs(i, j + 1)`. Lấy nhỏ hơn.
- Nếu `i` hết chuỗi: phải xóa hết `n - j` ký tự còn lại của `word2` (và ngược lại).

Cách này đúng nhưng mỗi cặp `(i, j)` bị tính đi tính lại rất nhiều lần.

In [8]:
def min_distance_brute(word1: str, word2: str) -> int:
    m, n = len(word1), len(word2)

    def dfs(i: int, j: int) -> int:
        if i == m:
            return n - j          # xóa hết phần còn lại của word2
        if j == n:
            return m - i          # xóa hết phần còn lại của word1
        if word1[i] == word2[j]:
            return dfs(i + 1, j + 1)
        return 1 + min(dfs(i + 1, j), dfs(i, j + 1))

    return dfs(0, 0)

for w1, w2, expected in examples:
    got = min_distance_brute(w1, w2)
    print(w1, w2, got, "OK" if got == expected else "SAI")
    assert got == expected

sea eat 2 OK
leetcode etco 4 OK


**Oracle độc lập.** Để kiểm chứng mọi cách sau này, ta cần một lời giải "chắc chắn đúng" dựa trực tiếp vào định nghĩa: thử **mọi dãy con** của `word1` (bằng bitmask), giữ lại cái nào cũng là dãy con của `word2` và dài nhất. Cách này không dùng DP nên không chia sẻ lỗi với các cách DP. Chỉ dùng cho chuỗi ngắn.

In [9]:
def oracle(word1: str, word2: str) -> int:
    m = len(word1)
    best = 0
    for mask in range(1 << m):
        sub = "".join(word1[k] for k in range(m) if mask >> k & 1)
        if len(sub) > best and is_subsequence(sub, word2):
            best = len(sub)
    return len(word1) + len(word2) - 2 * best

for w1, w2, expected in examples:
    assert oracle(w1, w2) == expected
print("oracle khớp các ví dụ")

oracle khớp các ví dụ


**Độ phức tạp và nút thắt.** Mỗi lần khác nhau, hàm rẽ **hai** nhánh, độ sâu tối đa `m + n`, nên trường hợp xấu là `O(2^(m+n))` thời gian (chính xác là cỡ số đường đi trên lưới), bộ nhớ `O(m + n)` cho ngăn xếp đệ quy. Hãy đếm số lần gọi khi hai chuỗi **không có ký tự chung** (xấu nhất) và so với số trạng thái thật sự khác nhau `(m + 1)(n + 1)`.

In [10]:
def count_calls(m: int, n: int) -> int:
    w1, w2 = "a" * m, "b" * n          # không có ký tự chung -> luôn rẽ 2 nhánh
    cnt = 0
    def dfs(i, j):
        nonlocal cnt
        cnt += 1
        if i == m:
            return n - j
        if j == n:
            return m - i
        return 1 + min(dfs(i + 1, j), dfs(i, j + 1))
    dfs(0, 0)
    return cnt

print(f"{'m=n':>4} {'số lần gọi':>12} {'số trạng thái (m+1)(n+1)':>26}")
for k in (2, 4, 6, 8, 10):
    print(f"{k:>4} {count_calls(k, k):>12} {(k + 1) * (k + 1):>26}")

 m=n   số lần gọi   số trạng thái (m+1)(n+1)
   2           11                          9
   4          139                         25
   6         1847                         49
   8        25739                         81
  10       369511                        121


Số lần gọi tăng **gấp bội** mỗi khi thêm ký tự, trong khi số trạng thái khác nhau chỉ tăng bậc hai. Với 500 x 500 thì đệ quy thuần là bất khả thi. Nút thắt: **các bài toán con lặp lại**.

## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát biến "bất khả thi" thành "dễ":

1. **Chỉ có `(m + 1)(n + 1)` trạng thái** `(i, j)`. Mỗi trạng thái cho đáp án cố định, nên tính **một lần** rồi nhớ lại (memoization) hoặc điền bảng theo thứ tự (bottom-up).
2. Đáp án có dạng đóng: `m + n - 2 * LCS`. Nghĩa là ta chỉ cần **độ dài** LCS, một bài DP rất chuẩn mà ta còn có thể tối ưu bộ nhớ và thậm chí đưa về LIS.

Từ đó có bốn cách "thật sự khác nhau về ý tưởng" nữa:

| Cách | Ý tưởng |
|---|---|
| 2 | Giữ nguyên đệ quy nhưng **nhớ** kết quả (top-down) |
| 3 | Điền bảng `f[i][j]` bằng vòng lặp (bottom-up), `f` là số lần xóa |
| 4 | Tính LCS bằng mảng 1 chiều rồi trả `m + n - 2*LCS` |
| 5 | Đổi LCS thành LIS trên danh sách vị trí, dùng `bisect` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ (top-down)

**Giải thích từng đoạn.**

- Dùng `@lru_cache(None)` bọc `dfs(i, j)` để mỗi cặp chỉ tính một lần.
- Thân hàm y hệt Cách 1.
- Lưu ý độ sâu đệ quy tối đa `m + n` (tới 1000) nên khi nộp nên nâng giới hạn đệ quy, nếu không có thể dính `RecursionError`.

In [11]:
def min_distance_memo(word1: str, word2: str) -> int:
    m, n = len(word1), len(word2)

    @lru_cache(None)
    def dfs(i: int, j: int) -> int:
        if i == m:
            return n - j
        if j == n:
            return m - i
        if word1[i] == word2[j]:
            return dfs(i + 1, j + 1)
        return 1 + min(dfs(i + 1, j), dfs(i, j + 1))

    return dfs(0, 0)

for w1, w2, expected in examples:
    assert min_distance_memo(w1, w2) == expected
print("memo OK; số trạng thái tối đa với 'leetcode','etco':", (8 + 1) * (4 + 1))

memo OK; số trạng thái tối đa với 'leetcode','etco': 45


**Code nộp LeetCode (Cách 2):**

In [12]:
class Solution:
    def minDistance(self, word1: str, word2: str) -> int:
        import sys
        from functools import lru_cache
        sys.setrecursionlimit(10000)          # độ sâu có thể tới m + n = 1000
        m, n = len(word1), len(word2)

        @lru_cache(None)
        def dfs(i: int, j: int) -> int:
            if i == m:
                return n - j
            if j == n:
                return m - i
            if word1[i] == word2[j]:
                return dfs(i + 1, j + 1)
            return 1 + min(dfs(i + 1, j), dfs(i, j + 1))

        return dfs(0, 0)

sol_memo = Solution().minDistance
print(sol_memo("sea", "eat"), sol_memo("leetcode", "etco"))

2 4


**Độ phức tạp (Cách 2):** thời gian `O(m * n)` (mỗi trạng thái một lần, mỗi lần `O(1)`), bộ nhớ `O(m * n)` cho bộ nhớ đệm cộng `O(m + n)` ngăn xếp.

---

### 7.2. Cách 3 — Bảng DP 2 chiều (bottom-up)

**Giải thích từng đoạn.** Đổi hướng định nghĩa sang **tiền tố**: `f[i][j]` = số lần xóa tối thiểu để `word1[:i]` và `word2[:j]` giống nhau.

- **Biên:** `f[i][0] = i` (xóa hết `i` ký tự của `word1`), `f[0][j] = j`.
- **Chuyển trạng thái:** nếu `word1[i-1] == word2[j-1]` thì `f[i][j] = f[i-1][j-1]`; ngược lại `f[i][j] = min(f[i-1][j], f[i][j-1]) + 1`.
- **Đáp án:** `f[m][n]`.
- Thứ tự điền: hàng tăng dần, mỗi hàng cột tăng dần, vì ô cần ba ô "phía trên-trái" đã có.

In [13]:
def min_distance_2d(word1: str, word2: str) -> int:
    m, n = len(word1), len(word2)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        f[i][0] = i
    for j in range(1, n + 1):
        f[0][j] = j
    for i, a in enumerate(word1, 1):
        for j, b in enumerate(word2, 1):
            if a == b:
                f[i][j] = f[i - 1][j - 1]
            else:
                f[i][j] = min(f[i - 1][j], f[i][j - 1]) + 1
    return f[m][n]

for w1, w2, expected in examples:
    assert min_distance_2d(w1, w2) == expected
print("2D OK")

2D OK


**Code nộp LeetCode (Cách 3):**

In [14]:
class Solution:
    def minDistance(self, word1: str, word2: str) -> int:
        m, n = len(word1), len(word2)
        f = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            f[i][0] = i
        for j in range(1, n + 1):
            f[0][j] = j
        for i, a in enumerate(word1, 1):
            for j, b in enumerate(word2, 1):
                if a == b:
                    f[i][j] = f[i - 1][j - 1]
                else:
                    f[i][j] = min(f[i - 1][j], f[i][j - 1]) + 1
        return f[m][n]

sol_2d = Solution().minDistance
print(sol_2d("sea", "eat"), sol_2d("leetcode", "etco"))

2 4


**Độ phức tạp (Cách 3):** thời gian `O(m * n)`, bộ nhớ `O(m * n)`. Với 500 x 500 là 251.001 ô — nhỏ.

---

### 7.3. Cách 4 — Dùng công thức LCS với mảng 1 chiều

**Giải thích từng đoạn.**

- Công thức: `đáp án = m + n - 2 * LCS`.
- LCS có bảng `g[i][j]` = độ dài LCS của hai tiền tố. Nếu hai ký tự cuối bằng nhau: `g[i][j] = g[i-1][j-1] + 1`; ngược lại `g[i][j] = max(g[i-1][j], g[i][j-1])`.
- Mỗi hàng `i` chỉ cần hàng `i - 1` -> **cuộn** thành một mảng `dp` độ dài `n + 1`. Khi cập nhật `dp[j]` ta cần giá trị chéo cũ `g[i-1][j-1]`, nên lưu nó vào biến `prev` trước khi bị ghi đè.
- Chọn chuỗi ngắn làm chiều của mảng để bộ nhớ là `O(min(m, n))`.

In [15]:
def lcs_1d(a: str, b: str) -> int:
    if len(b) > len(a):
        a, b = b, a                    # b là chuỗi ngắn hơn
    dp = [0] * (len(b) + 1)
    for ch in a:
        prev = 0                       # g[i-1][j-1], bắt đầu bằng g[i-1][0] = 0
        for j in range(1, len(b) + 1):
            tmp = dp[j]                # g[i-1][j] cũ, sẽ thành "chéo" cho j + 1
            if ch == b[j - 1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = tmp
    return dp[-1]

def min_distance_lcs1d(word1: str, word2: str) -> int:
    return len(word1) + len(word2) - 2 * lcs_1d(word1, word2)

print("LCS(sea, eat) =", lcs_1d("sea", "eat"), "| LCS(leetcode, etco) =", lcs_1d("leetcode", "etco"))
for w1, w2, expected in examples:
    assert min_distance_lcs1d(w1, w2) == expected
print("LCS 1D OK")

LCS(sea, eat) = 2 | LCS(leetcode, etco) = 4
LCS 1D OK


**Code nộp LeetCode (Cách 4):**

In [16]:
class Solution:
    def minDistance(self, word1: str, word2: str) -> int:
        a, b = word1, word2
        if len(b) > len(a):
            a, b = b, a
        dp = [0] * (len(b) + 1)
        for ch in a:
            prev = 0
            for j in range(1, len(b) + 1):
                tmp = dp[j]
                if ch == b[j - 1]:
                    dp[j] = prev + 1
                else:
                    dp[j] = max(dp[j], dp[j - 1])
                prev = tmp
        return len(word1) + len(word2) - 2 * dp[-1]

sol_lcs1d = Solution().minDistance
print(sol_lcs1d("sea", "eat"), sol_lcs1d("leetcode", "etco"))

2 4


**Độ phức tạp (Cách 4):** thời gian `O(m * n)`, bộ nhớ `O(min(m, n))`.

---

### 7.4. Cách 5 — LCS bằng danh sách vị trí + `bisect` (đưa về LIS)

**Ý tưởng khác hẳn.** Với mỗi ký tự, lưu **danh sách vị trí** của nó trong `word2`. Duyệt `word1` từ trái sang phải; mỗi ký tự `ch` của `word1` cho ta các vị trí có thể ghép trong `word2`. Một dãy con chung tương ứng với một dãy vị trí trong `word2` **tăng ngặt** (LIS - Longest Increasing Subsequence).

**Giải thích từng đoạn.**

- `pos[ch]` = các chỉ số của `ch` trong `word2`, tăng dần.
- Mảng `tails`: `tails[k]` = vị trí cuối **nhỏ nhất** có thể của một dãy tăng độ dài `k + 1` (kỹ thuật patience sorting).
- Với mỗi ký tự `ch` của `word1`, duyệt `pos[ch]` theo thứ tự **giảm dần** rồi cập nhật `tails` bằng `bisect_left`. Duyệt giảm dần đảm bảo một ký tự của `word1` không bị dùng hai lần trong cùng một dãy (vì vị trí đã dùng lớn hơn sẽ không "đứng trước" vị trí nhỏ hơn nữa).
- `len(tails)` = LCS.

In [17]:
def lcs_lis(a: str, b: str) -> int:
    pos = defaultdict(list)
    for j, ch in enumerate(b):
        pos[ch].append(j)
    tails = []
    for ch in a:
        for j in reversed(pos.get(ch, [])):
            k = bisect_left(tails, j)
            if k == len(tails):
                tails.append(j)
            else:
                tails[k] = j
    return len(tails)

def min_distance_lis(word1: str, word2: str) -> int:
    return len(word1) + len(word2) - 2 * lcs_lis(word1, word2)

print("LCS(sea, eat) =", lcs_lis("sea", "eat"))
for w1, w2, expected in examples:
    assert min_distance_lis(w1, w2) == expected
print("LIS-bisect OK")

LCS(sea, eat) = 2
LIS-bisect OK


**Code nộp LeetCode (Cách 5):**

In [18]:
class Solution:
    def minDistance(self, word1: str, word2: str) -> int:
        from bisect import bisect_left
        from collections import defaultdict
        pos = defaultdict(list)
        for j, ch in enumerate(word2):
            pos[ch].append(j)
        tails = []
        for ch in word1:
            for j in reversed(pos.get(ch, [])):
                k = bisect_left(tails, j)
                if k == len(tails):
                    tails.append(j)
                else:
                    tails[k] = j
        return len(word1) + len(word2) - 2 * len(tails)

sol_lis = Solution().minDistance
print(sol_lis("sea", "eat"), sol_lis("leetcode", "etco"))

2 4


**Độ phức tạp (Cách 5):** gọi `r` là số cặp `(i, j)` có `word1[i] == word2[j]`. Thời gian `O(r log n)` (xấu nhất `r = m * n` khi cả hai chuỗi toàn một chữ cái -> `O(m n log n)`, chậm hơn Cách 3 trong tình huống đó; nhưng khi bảng chữ cái lớn thì `r` nhỏ và rất nhanh). Bộ nhớ `O(n)`.

## 8. TRACE từng bước

### 8.1. Bảng DP cho `word1 = "sea"`, `word2 = "eat"`

Hàng ứng với tiền tố của `word1`, cột ứng với tiền tố của `word2`. Cách đọc ô `f[i][j]`: "xóa tối thiểu bao nhiêu ký tự để `word1[:i]` và `word2[:j]` bằng nhau".

| | "" | e | ea | eat |
|---|---|---|---|---|
| "" | 0 | 1 | 2 | 3 |
| s | 1 | 2 | 3 | 4 |
| se | 2 | 1 | 2 | 3 |
| sea | 3 | 2 | 1 | 2 |

Ô cuối `f[3][3] = 2` chính là đáp án. Ô `f[3][2] = 1` đọc là: `"sea"` và `"ea"` cần xóa 1 ký tự (`s`) để giống nhau.

Hàm `trace_dp` dưới đây in từng bước điền để bạn đối chiếu.

In [19]:
def show_table(f, word1, word2):
    header = "      " + "  ".join(f"{c:>2}" for c in ["_"] + list(word2))
    print(header)
    for i, row in enumerate(f):
        label = "_" if i == 0 else word1[i - 1]
        print(f"  {label:>2}  " + "  ".join(f"{v:>2}" for v in row))

def trace_dp(word1: str, word2: str) -> int:
    m, n = len(word1), len(word2)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        f[i][0] = i
    for j in range(1, n + 1):
        f[0][j] = j
    print("Sau khi khởi tạo biên:")
    show_table(f, word1, word2)
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            a, b = word1[i - 1], word2[j - 1]
            if a == b:
                f[i][j] = f[i - 1][j - 1]
                print(f"f[{i}][{j}]: '{a}' == '{b}' -> giữ cả hai, f = f[{i-1}][{j-1}] = {f[i][j]}")
            else:
                up, left = f[i - 1][j], f[i][j - 1]
                f[i][j] = min(up, left) + 1
                side = "xóa ký tự của word1" if up <= left else "xóa ký tự của word2"
                print(f"f[{i}][{j}]: '{a}' != '{b}' -> min(trên={up}, trái={left}) + 1 = {f[i][j]}  ({side})")
    print("Bảng cuối:")
    show_table(f, word1, word2)
    return f[m][n]

ans = trace_dp("sea", "eat")
print("đáp án =", ans)
assert ans == 2

Sau khi khởi tạo biên:
       _   e   a   t
   _   0   1   2   3
   s   1   0   0   0
   e   2   0   0   0
   a   3   0   0   0
f[1][1]: 's' != 'e' -> min(trên=1, trái=1) + 1 = 2  (xóa ký tự của word1)
f[1][2]: 's' != 'a' -> min(trên=2, trái=2) + 1 = 3  (xóa ký tự của word1)
f[1][3]: 's' != 't' -> min(trên=3, trái=3) + 1 = 4  (xóa ký tự của word1)
f[2][1]: 'e' == 'e' -> giữ cả hai, f = f[1][0] = 1
f[2][2]: 'e' != 'a' -> min(trên=3, trái=1) + 1 = 2  (xóa ký tự của word2)
f[2][3]: 'e' != 't' -> min(trên=4, trái=2) + 1 = 3  (xóa ký tự của word2)
f[3][1]: 'a' != 'e' -> min(trên=1, trái=3) + 1 = 2  (xóa ký tự của word1)
f[3][2]: 'a' == 'a' -> giữ cả hai, f = f[2][1] = 1
f[3][3]: 'a' != 't' -> min(trên=3, trái=1) + 1 = 2  (xóa ký tự của word2)
Bảng cuối:
       _   e   a   t
   _   0   1   2   3
   s   1   2   3   4
   e   2   1   2   3
   a   3   2   1   2
đáp án = 2


### 8.2. Truy vết: xóa những ký tự nào?

Bảng DP còn cho ta biết **chuỗi còn lại** và **các vị trí bị xóa**. Đi ngược từ `f[m][n]`: nếu hai ký tự bằng nhau thì đi chéo (giữ); nếu không, đi về ô nhỏ hơn trong hai ô (trên/trái) — nghĩa là xóa ký tự tương ứng.

In [20]:
def reconstruct(word1: str, word2: str):
    m, n = len(word1), len(word2)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        f[i][0] = i
    for j in range(1, n + 1):
        f[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                f[i][j] = f[i - 1][j - 1]
            else:
                f[i][j] = min(f[i - 1][j], f[i][j - 1]) + 1
    i, j = m, n
    kept, del1, del2 = [], [], []
    while i > 0 and j > 0:
        if word1[i - 1] == word2[j - 1]:
            kept.append(word1[i - 1]); i -= 1; j -= 1
        elif f[i - 1][j] <= f[i][j - 1]:
            del1.append(i - 1); i -= 1
        else:
            del2.append(j - 1); j -= 1
    while i > 0:
        del1.append(i - 1); i -= 1
    while j > 0:
        del2.append(j - 1); j -= 1
    return "".join(reversed(kept)), sorted(del1), sorted(del2), f[m][n]

for w1, w2, expected in examples:
    kept, d1, d2, steps = reconstruct(w1, w2)
    print(f"{w1} / {w2}: giữ {kept!r}; xóa vị trí {d1} của word1 và {d2} của word2; tổng {steps}")
    assert len(d1) + len(d2) == steps == expected
    assert is_subsequence(kept, w1) and is_subsequence(kept, w2)
    assert len(w1) - len(d1) == len(kept) == len(w2) - len(d2)

sea / eat: giữ 'ea'; xóa vị trí [0] của word1 và [2] của word2; tổng 2
leetcode / etco: giữ 'etco'; xóa vị trí [0, 1, 6, 7] của word1 và [] của word2; tổng 4


### 8.3. Trace mảng 1 chiều của Cách 4

Với `a = "sea"` (duyệt) và `b = "eat"` (làm chiều mảng), ta xem `dp` thay đổi qua từng ký tự của `a`. `dp[j]` ở cuối hàng `i` là LCS của `a[:i]` với `b[:j]`.

In [21]:
def trace_lcs_1d(a: str, b: str) -> int:
    dp = [0] * (len(b) + 1)
    print("b =", list(b), "| dp ban đầu =", dp)
    for i, ch in enumerate(a, 1):
        prev = 0
        for j in range(1, len(b) + 1):
            tmp = dp[j]
            if ch == b[j - 1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = tmp
        print(f"sau ký tự a[{i-1}]={ch!r}: dp = {dp}")
    return dp[-1]

lcs = trace_lcs_1d("sea", "eat")
print("LCS =", lcs, "-> đáp án =", 3 + 3 - 2 * lcs)
assert 3 + 3 - 2 * lcs == 2

b = ['e', 'a', 't'] | dp ban đầu = [0, 0, 0, 0]
sau ký tự a[0]='s': dp = [0, 0, 0, 0]
sau ký tự a[1]='e': dp = [0, 1, 1, 1]
sau ký tự a[2]='a': dp = [0, 1, 2, 2]
LCS = 2 -> đáp án = 2


### 8.4. Trace Cách 5 (bisect)

Với `a = "sea"`, `b = "eat"`: `pos = {e: [0], a: [1], t: [2]}`. Ta xem mảng `tails` thay đổi.

In [22]:
def trace_lis(a: str, b: str) -> int:
    pos = defaultdict(list)
    for j, ch in enumerate(b):
        pos[ch].append(j)
    print("pos =", dict(pos))
    tails = []
    for ch in a:
        for j in reversed(pos.get(ch, [])):
            k = bisect_left(tails, j)
            action = "thêm vào cuối" if k == len(tails) else f"ghi đè tails[{k}]"
            if k == len(tails):
                tails.append(j)
            else:
                tails[k] = j
            print(f"ký tự {ch!r}, vị trí {j}: {action} -> tails = {tails}")
        if not pos.get(ch):
            print(f"ký tự {ch!r}: không có trong b, bỏ qua -> tails = {tails}")
    return len(tails)

print("LCS =", trace_lis("sea", "eat"))
print()
print("LCS =", trace_lis("aab", "aaab"))

pos = {'e': [0], 'a': [1], 't': [2]}
ký tự 's': không có trong b, bỏ qua -> tails = []
ký tự 'e', vị trí 0: thêm vào cuối -> tails = [0]
ký tự 'a', vị trí 1: thêm vào cuối -> tails = [0, 1]
LCS = 2

pos = {'a': [0, 1, 2], 'b': [3]}
ký tự 'a', vị trí 2: thêm vào cuối -> tails = [2]
ký tự 'a', vị trí 1: ghi đè tails[0] -> tails = [1]
ký tự 'a', vị trí 0: ghi đè tails[0] -> tails = [0]
ký tự 'a', vị trí 2: thêm vào cuối -> tails = [0, 2]
ký tự 'a', vị trí 1: ghi đè tails[1] -> tails = [0, 1]
ký tự 'a', vị trí 0: ghi đè tails[0] -> tails = [0, 1]
ký tự 'b', vị trí 3: thêm vào cuối -> tails = [0, 1, 3]
LCS = 3


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (bảng DP 2 chiều) làm code nộp chính: ngắn, dễ kiểm, không đệ quy nên không lo `RecursionError`, thừa nhanh cho 500 x 500.

```python
class Solution:
    def minDistance(self, word1: str, word2: str) -> int:
        m, n = len(word1), len(word2)
        f = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            f[i][0] = i
        for j in range(1, n + 1):
            f[0][j] = j
        for i, a in enumerate(word1, 1):
            for j, b in enumerate(word2, 1):
                if a == b:
                    f[i][j] = f[i - 1][j - 1]
                else:
                    f[i][j] = min(f[i - 1][j], f[i][j - 1]) + 1
        return f[m][n]
```

In [23]:
class Solution:
    def minDistance(self, word1: str, word2: str) -> int:
        m, n = len(word1), len(word2)
        f = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            f[i][0] = i
        for j in range(1, n + 1):
            f[0][j] = j
        for i, a in enumerate(word1, 1):
            for j, b in enumerate(word2, 1):
                if a == b:
                    f[i][j] = f[i - 1][j - 1]
                else:
                    f[i][j] = min(f[i - 1][j], f[i][j - 1]) + 1
        return f[m][n]

sol = Solution()
for w1, w2, expected in examples:
    got = sol.minDistance(w1, w2)
    print(w1, w2, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

sea eat -> 2 OK
leetcode etco -> 4 OK


## 10. Vì sao đúng

**(a) Công thức `m + n - 2 * LCS`.** Gọi `T` là chuỗi cuối cùng. `T` là dãy con của cả hai chuỗi nên `len(T) <= LCS`. Số bước là `m + n - 2 * len(T)` nên nhỏ nhất khi `len(T) = LCS`. Ngược lại, lấy một LCS bất kỳ làm `T` thì xóa các ký tự không thuộc nó ở mỗi chuỗi là hợp lệ. Hai chiều cùng dẫn tới `LCS`.

**(b) Công thức chuyển trạng thái của `f`.** Xét `f[i][j]` với hai ký tự cuối `a = word1[i-1]`, `b = word2[j-1]`.

- Nếu `a == b`: tồn tại cách tối ưu giữ cả hai ký tự cuối. Giả sử cách tối ưu xóa `a` hoặc `b` (hoặc cả hai) — ta có thể đổi lấy cách giữ cặp này mà không tăng số xóa (nếu `a` bị xóa còn `b` được ghép với một ký tự `a'` khác trước nó, thì ghép `a` với `b` thay vì `a'` cũng hợp lệ, vì `a` đứng sau `a'`; ví dụ tương tự khi `b` bị xóa). Vậy `f[i][j] = f[i-1][j-1]`.
- Nếu `a != b`: hai ký tự cuối không thể được ghép với nhau, nên ít nhất một trong hai bị xóa. Xóa `a` tốn 1 + `f[i-1][j]`, xóa `b` tốn 1 + `f[i][j-1]`. Cách tối ưu là cái nhỏ hơn.
- Biên: một chuỗi rỗng thì phải xóa hết chuỗi kia.

**(c) Cách 5.** LCS bằng độ dài dãy tăng ngặt dài nhất của vị trí: mỗi dãy con chung ứng với một bộ vị trí `j_1 < j_2 < ...` trong `word2` khớp lần lượt với `word1`. Duyệt vị trí giảm dần cho mỗi ký tự của `word1` bảo đảm không dùng cùng một ký tự của `word1` hai lần.

**Kiểm chứng bằng số:** ta để máy xác nhận (b) bằng cách so sánh `f[m][n]` với oracle trên mọi cặp chuỗi nhị phân ngắn.

In [24]:
def all_strings(alphabet: str, max_len: int):
    out = [""]
    frontier = [""]
    for _ in range(max_len):
        frontier = [s + c for s in frontier for c in alphabet]
        out += frontier
    return out

strings = [s for s in all_strings("ab", 5) if s]   # 62 chuỗi độ dài 1..5
checked = 0
for s in strings:
    for t in strings:
        assert min_distance_2d(s, t) == oracle(s, t), (s, t)
        checked += 1
print("so khớp công thức DP với oracle trên", checked, "cặp chuỗi")

so khớp công thức DP với oracle trên 3844 cặp chuỗi


## 11. Time / Space Complexity

Gọi `m = len(word1)`, `n = len(word2)`.

| Cách | Time | Space |
|---|---|---|
| 1. Đệ quy thuần | `O(2^(m+n))` | `O(m + n)` |
| 2. Đệ quy có nhớ | `O(mn)` | `O(mn)` |
| 3. Bảng DP 2 chiều | `O(mn)` | `O(mn)` |
| 4. LCS mảng 1 chiều | `O(mn)` | `O(min(m, n))` |
| 5. LCS bằng LIS/bisect | `O(r log n)`, xấu nhất `O(mn log n)` | `O(n)` |

Với `m = n = 500`, `mn = 250.000` phép so sánh: Python chạy cỡ vài chục mili-giây. Hãy đo để thấy.

In [25]:
random.seed(583)
big1 = "".join(random.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(500))
big2 = "".join(random.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(500))
bigA1 = "".join(random.choice("ab") for _ in range(500))
bigA2 = "".join(random.choice("ab") for _ in range(500))

for name, fn in [("2D", min_distance_2d), ("LCS 1D", min_distance_lcs1d), ("LIS bisect", min_distance_lis), ("memo", sol_memo)]:
    for tag, (x, y) in [("26 chữ cái", (big1, big2)), ("2 chữ cái", (bigA1, bigA2))]:
        t0 = time.perf_counter()
        r = fn(x, y)
        dt = (time.perf_counter() - t0) * 1000
        print(f"{name:11} {tag:11} -> {r:4}  {dt:7.1f} ms")

2D          26 chữ cái  ->  696     39.0 ms


2D          2 chữ cái   ->  180     26.7 ms


LCS 1D      26 chữ cái  ->  696     34.3 ms
LCS 1D      2 chữ cái   ->  180     29.3 ms
LIS bisect  26 chữ cái  ->  696      1.4 ms
LIS bisect  2 chữ cái   ->  180     17.1 ms


memo        26 chữ cái  ->  696    273.1 ms
memo        2 chữ cái   ->  180     77.5 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách làm sai, đối chiếu tất cả các cách với nhau và với oracle, và test ngẫu nhiên có seed cố định.

In [26]:
solutions = {
    "brute": min_distance_brute,
    "memo": sol_memo,
    "2d": sol_2d,
    "lcs1d": sol_lcs1d,
    "lis": sol_lis,
    "final": Solution().minDistance,
}

# 12.1 ví dụ đề
for w1, w2, expected in examples:
    for name, fn in solutions.items():
        assert fn(w1, w2) == expected, (name, w1, w2)
print("12.1 ví dụ đề: OK")

# 12.2 edge case
edge = [
    ("a", "a", 0),                 # bằng nhau, độ dài 1
    ("a", "b", 2),                 # không chung gì -> xóa cả hai
    ("abc", "abc", 0),             # bằng nhau
    ("abc", "def", 6),             # hoàn toàn khác
    ("abc", "a", 2),               # một chuỗi là dãy con
    ("a", "abc", 2),               # đối xứng
    ("abc", "c", 2),
    ("aaaa", "aa", 2),             # lặp ký tự
    ("ab", "ba", 2),               # đảo thứ tự: chỉ giữ được 1 ký tự
    ("abcd", "dcba", 6),           # đảo ngược: LCS = 1
    ("aab", "aaab", 1),
    ("a" * 7, "a" * 3, 4),
]
for w1, w2, expected in edge:
    for name, fn in solutions.items():
        assert fn(w1, w2) == expected, (name, w1, w2, fn(w1, w2), expected)
print("12.2 edge case: OK", len(edge), "ca")

# 12.3 phản ví dụ: các cách "gần đúng" nhưng sai
def wrong_counter(w1, w2):
    # đếm ký tự chung (bỏ qua thứ tự): sai
    common = sum((Counter(w1) & Counter(w2)).values())
    return len(w1) + len(w2) - 2 * common

def wrong_edit_like(w1, w2):
    # nhầm với Edit Distance: mismatch thì f[i-1][j-1] + 1
    m, n = len(w1), len(w2)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): f[i][0] = i
    for j in range(n + 1): f[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if w1[i-1] == w2[j-1]:
                f[i][j] = f[i-1][j-1]
            else:
                f[i][j] = min(f[i-1][j], f[i][j-1], f[i-1][j-1]) + 1
    return f[m][n]

print("ab/ba: đúng =", min_distance_2d("ab", "ba"), "| đếm-ký-tự-chung =", wrong_counter("ab", "ba"))
print("a/b  : đúng =", min_distance_2d("a", "b"), "| kiểu Edit Distance =", wrong_edit_like("a", "b"))
assert wrong_counter("ab", "ba") != 2
assert wrong_edit_like("a", "b") != 2
print("12.3 phản ví dụ: các cách sai bị lộ")

# 12.4 đối chiếu mọi cặp chuỗi nhị phân độ dài <= 4 với oracle, mọi cách
small = [s for s in all_strings("ab", 4) if s]
for s in small:
    for t in small:
        ref = oracle(s, t)
        for name, fn in solutions.items():
            assert fn(s, t) == ref, (name, s, t)
print("12.4 đối chiếu toàn bộ chuỗi nhị phân độ dài <= 4:", len(small) ** 2, "cặp x", len(solutions), "cách: OK")

# 12.5 ngẫu nhiên có seed, đúng ràng buộc (độ dài nhỏ để oracle chạy được)
random.seed(2024)
for _ in range(400):
    m = random.randint(1, 9)
    n = random.randint(1, 12)
    alpha = random.choice(["ab", "abc", "abcdef"])
    s = "".join(random.choice(alpha) for _ in range(m))
    t = "".join(random.choice(alpha) for _ in range(n))
    ref = oracle(s, t)
    for name, fn in solutions.items():
        assert fn(s, t) == ref, (name, s, t)
print("12.5 ngẫu nhiên nhỏ so với oracle: OK")

# 12.6 ngẫu nhiên lớn (đến 500): các cách DP đối chiếu nhau
random.seed(7)
fast = {k: v for k, v in solutions.items() if k not in ("brute",)}
for _ in range(15):
    m = random.randint(1, 500)
    n = random.randint(1, 500)
    alpha = random.choice(["ab", "abc", "abcdefghijklmnopqrstuvwxyz"])
    s = "".join(random.choice(alpha) for _ in range(m))
    t = "".join(random.choice(alpha) for _ in range(n))
    res = {name: fn(s, t) for name, fn in fast.items()}
    assert len(set(res.values())) == 1, res
    assert 0 <= res["2d"] <= m + n
print("12.6 ngẫu nhiên lớn (<= 500): các cách khớp nhau")

# 12.7 giá trị cực trị
assert Solution().minDistance("a" * 500, "a" * 500) == 0
assert Solution().minDistance("a" * 500, "b" * 500) == 1000
assert Solution().minDistance("a" * 500, "a") == 499
print("12.7 cực trị: OK")

12.1 ví dụ đề: OK
12.2 edge case: OK 12 ca
ab/ba: đúng = 2 | đếm-ký-tự-chung = 0
a/b  : đúng = 2 | kiểu Edit Distance = 1
12.3 phản ví dụ: các cách sai bị lộ
12.4 đối chiếu toàn bộ chuỗi nhị phân độ dài <= 4: 900 cặp x 6 cách: OK


12.5 ngẫu nhiên nhỏ so với oracle: OK


12.6 ngẫu nhiên lớn (<= 500): các cách khớp nhau
12.7 cực trị: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Khởi tạo biên bằng 0

Rất nhiều người để `f` toàn số 0 rồi quên `f[i][0] = i`, `f[0][j] = j`. Hệ quả: khi một chuỗi hết trước, ta tưởng không cần xóa phần dư của chuỗi kia.

In [27]:
# SAI: không đặt biên
def wrong_no_border(word1, word2):
    m, n = len(word1), len(word2)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                f[i][j] = f[i - 1][j - 1]
            else:
                f[i][j] = min(f[i - 1][j], f[i][j - 1]) + 1
    return f[m][n]

print("sai :", wrong_no_border("abc", "a"), "(đáp án thật 2)")
print("đúng:", min_distance_2d("abc", "a"))
assert wrong_no_border("abc", "a") != 2 and min_distance_2d("abc", "a") == 2

sai : 1 (đáp án thật 2)
đúng: 2


Vì sao sai: `f[3][0]` phải bằng 3 (xóa cả `"abc"` để bằng `""`). Để 0 thì mọi ô dựa vào biên đều bị thiếu.

### Lỗi 2 — Nhầm với Edit Distance (đi chéo cộng 1)

Trong Edit Distance (72) có thể **thay** một ký tự, nên khi khác nhau có thêm lựa chọn `f[i-1][j-1] + 1`. Ở bài này thay thế = xóa 2 (xóa mỗi bên một), tức `f[i-1][j-1] + 2`, và nó **không bao giờ** tốt hơn `min(trên, trái) + 1`. Dùng `+ 1` cho đường chéo là sai (đã thấy ở `wrong_edit_like` mục 12.3: với `"a"`, `"b"` nó cho 1 thay vì 2).

In [28]:
print("a/b bài này:", min_distance_2d("a", "b"), "| code nhầm kiểu Edit Distance:", wrong_edit_like("a", "b"))
assert min_distance_2d("a", "b") == 2 and wrong_edit_like("a", "b") == 1
print("sea/eat bài này:", min_distance_2d("sea", "eat"), "| kiểu Edit Distance:", wrong_edit_like("sea", "eat"))
assert wrong_edit_like("sea", "eat") == 2   # ví dụ này tình cờ trùng; "a"/"b" mới lộ lỗi
r = wrong_edit_like("abcd", "dcba")
print("abcd/dcba bài này:", min_distance_2d("abcd", "dcba"), "| kiểu Edit Distance:", r)
assert r != min_distance_2d("abcd", "dcba")

a/b bài này: 2 | code nhầm kiểu Edit Distance: 1
sea/eat bài này: 2 | kiểu Edit Distance: 2
abcd/dcba bài này: 6 | kiểu Edit Distance: 4


### Lỗi 3 — Trả về LCS thay vì số lần xóa

Công thức là `m + n - 2 * LCS`. Trả `LCS` hoặc `max(m, n) - LCS` đều sai.

In [29]:
def wrong_return_lcs(w1, w2):
    return lcs_1d(w1, w2)

def wrong_single_lcs(w1, w2):
    return len(w1) + len(w2) - lcs_1d(w1, w2)       # quên nhân đôi

print("abc/a: LCS =", wrong_return_lcs("abc", "a"), "| m+n-LCS =", wrong_single_lcs("abc", "a"), "| đúng =", min_distance_lcs1d("abc", "a"))
assert wrong_return_lcs("abc", "a") != 2
assert wrong_single_lcs("abc", "a") != 2
assert min_distance_lcs1d("abc", "a") == 2

abc/a: LCS = 1 | m+n-LCS = 3 | đúng = 2


### Lỗi 4 — `[[0] * (n + 1)] * (m + 1)`

Các hàng là **một** list. Ghi vào `f[i][0]` cho mọi `i` thì ô cuối cùng ghi đè tất cả. Kết quả bảng toàn giá trị lạ.

In [30]:
def wrong_alias(word1, word2):
    m, n = len(word1), len(word2)
    f = [[0] * (n + 1)] * (m + 1)       # SAI: các hàng dùng chung
    for i in range(1, m + 1):
        f[i][0] = i
    for j in range(1, n + 1):
        f[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                f[i][j] = f[i - 1][j - 1]
            else:
                f[i][j] = min(f[i - 1][j], f[i][j - 1]) + 1
    return f[m][n]

print("sai :", wrong_alias("sea", "eat"), "| đúng:", min_distance_2d("sea", "eat"))
assert wrong_alias("sea", "eat") != 2

sai : 5 | đúng: 2


### Lỗi 5 — Đếm ký tự chung mà bỏ qua thứ tự

Nhiều người thấy "xóa để hai chuỗi bằng nhau" liền dùng `Counter`: ký tự nào thừa thì xóa. Nhưng hai chuỗi bằng nhau cần **cùng thứ tự**, không chỉ cùng tập ký tự.

In [31]:
print("ab/ba: Counter nói", wrong_counter("ab", "ba"), "bước, thật ra cần", min_distance_2d("ab", "ba"))
print("abcd/dcba: Counter nói", wrong_counter("abcd", "dcba"), "bước, thật ra cần", min_distance_2d("abcd", "dcba"))
assert wrong_counter("abcd", "dcba") == 0 and min_distance_2d("abcd", "dcba") == 6

ab/ba: Counter nói 0 bước, thật ra cần 2
abcd/dcba: Counter nói 0 bước, thật ra cần 6


## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** hai chuỗi, thao tác trên mỗi chuỗi, hỏi số bước tối thiểu -> hãy nghĩ tới bảng `f[i][j]` theo tiền tố.
- **Chỉ xóa** (không chèn/thay) -> luôn quy về LCS: `m + n - 2 * LCS`. **Chỉ xóa ở một chuỗi** (biến `word1` thành `word2`) -> `m - LCS` và `word2` phải là dãy con.
- **Có thay thế** -> Edit Distance (72). **Có trọng số theo ký tự** (ví dụ tổng ASCII) -> DP cùng khung nhưng cộng giá trị ký tự (712).
- Khi bảng chỉ dựa vào hàng trên và ô chéo, hãy nghĩ ngay tới **mảng cuộn** để giảm bộ nhớ.
- Với bảng chữ cái lớn, cách `bisect` (Hunt-Szymanski) hay vượt trội vì chỉ xét các cặp ký tự khớp.
- Tự kiểm tra: luôn thử `("ab", "ba")`, `("a", "b")`, một chuỗi là dãy con của chuỗi kia.
- Nhớ **hàng/cột 0** là biên (chuỗi rỗng), ký tự thật là `word[i - 1]`.

## 15. Biến thể

Bốn bài cùng họ, giải bằng cùng khung. Dưới đây code ngắn và kiểm chứng bằng oracle vét cạn.

1. **LCS (1143)**: chỉ cần độ dài dãy con chung dài nhất — chính là `lcs_1d`.
2. **Edit Distance (72)**: cho phép chèn, xóa, thay; mỗi thao tác 1 bước.
3. **Shortest Common Supersequence (độ dài)** (liên quan 1092): chuỗi ngắn nhất có cả hai chuỗi là dãy con. Độ dài = `m + n - LCS`.
4. **Minimum ASCII Delete Sum (712)**: như bài này nhưng mỗi lần xóa tốn bằng mã ASCII của ký tự; tối thiểu tổng chi phí (bài tập 5).

In [32]:
def edit_distance(a: str, b: str) -> int:
    m, n = len(a), len(b)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): f[i][0] = i
    for j in range(n + 1): f[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                f[i][j] = f[i - 1][j - 1]
            else:
                f[i][j] = min(f[i - 1][j], f[i][j - 1], f[i - 1][j - 1]) + 1
    return f[m][n]

def scs_len(a: str, b: str) -> int:
    return len(a) + len(b) - lcs_1d(a, b)

# oracle cho Edit Distance: BFS trên các chuỗi (chỉ dùng cho chuỗi cực ngắn)
def edit_oracle(a: str, b: str) -> int:
    alphabet = sorted(set(a + b))
    from collections import deque
    seen = {a}
    dq = deque([(a, 0)])
    while dq:
        s, d = dq.popleft()
        if s == b:
            return d
        nxt = []
        for i in range(len(s) + 1):
            for c in alphabet:
                nxt.append(s[:i] + c + s[i:])            # chèn
        for i in range(len(s)):
            nxt.append(s[:i] + s[i + 1:])                # xóa
            for c in alphabet:
                nxt.append(s[:i] + c + s[i + 1:])        # thay
        for t in nxt:
            if t not in seen and len(t) <= len(a) + len(b):
                seen.add(t)
                dq.append((t, d + 1))
    return -1

print("LCS(sea, eat) =", lcs_1d("sea", "eat"))
print("Edit Distance(horse, ros) =", edit_distance("horse", "ros"))
print("SCS length(sea, eat) =", scs_len("sea", "eat"), "(vd: 'seat' chứa cả 'sea' và 'eat' như dãy con)")
assert edit_distance("horse", "ros") == 3
assert edit_distance("intention", "execution") == 5
assert scs_len("abac", "cab") == 5
for s in ["", "a", "ab", "abc", "ba"]:
    for t in ["", "a", "b", "ab", "bca"]:
        assert edit_distance(s, t) == edit_oracle(s, t), (s, t)
print("Biến thể: OK")

LCS(sea, eat) = 2
Edit Distance(horse, ros) = 3
SCS length(sea, eat) = 4 (vd: 'seat' chứa cả 'sea' và 'eat' như dãy con)
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy thuần | Rẽ nhánh theo ký tự đầu | `O(2^(m+n))` | `O(m+n)` | Chỉ để hiểu đề, chuỗi rất ngắn |
| 2. Đệ quy có nhớ | Như 1 nhưng nhớ `(i, j)` | `O(mn)` | `O(mn)` | Khi dễ nghĩ đệ quy; coi chừng giới hạn đệ quy |
| 3. Bảng 2D | Điền `f[i][j]` theo tiền tố | `O(mn)` | `O(mn)` | **Cách nộp chính**: rõ ràng, không đệ quy |
| 4. LCS 1D | `m + n - 2*LCS` với mảng cuộn | `O(mn)` | `O(min(m,n))` | Khi cần tiết kiệm bộ nhớ |
| 5. LIS + bisect | Vị trí ký tự + patience | `O(r log n)` | `O(n)` | Bảng chữ cái lớn, ít cặp khớp |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(min_distance_2d("abc", "abc"))
print(min_distance_2d("abc", "def"))
print(min_distance_2d("aaa", "aaaaa"))
```

In [33]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay bảng DP (dễ-vừa)

Với `word1 = "ab"`, `word2 = "ba"`, hãy điền tay bảng `f` kích thước 3 x 3 (hàng = tiền tố của `word1`, cột = tiền tố của `word2`, tính cả hàng/cột rỗng) dưới dạng list các list, và ghi đáp án.

In [34]:
# Bài 2: điền kết quả tính tay
hand_table = None    # ví dụ [[0, 1, 2], [...], [...]]
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm bảng DP.

In [35]:
def exercise_3_dp(word1: str, word2: str) -> int:
    m, n = len(word1), len(word2)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    # TODO 1: đặt biên f[i][0] và f[0][j]
    # TODO 2: hai vòng lặp i, j; nếu word1[i-1] == word2[j-1] thì sao chép ô chéo
    # TODO 3: ngược lại lấy min(ô trên, ô trái) + 1
    # TODO 4: trả về ô cuối
    return None


# assert exercise_3_dp("sea", "eat") == 2
# assert exercise_3_dp("leetcode", "etco") == 4
# assert exercise_3_dp("a", "b") == 2

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_lcs_form(word1, word2)` dùng công thức `m + n - 2 * LCS` với mảng 1 chiều.

In [36]:
def exercise_4_lcs_form(word1: str, word2: str) -> int:
    # TODO
    pass


# assert exercise_4_lcs_form("sea", "eat") == 2
# assert exercise_4_lcs_form("leetcode", "etco") == 4
# assert exercise_4_lcs_form("ab", "ba") == 2

### Bài 5 — Mini challenge: Minimum ASCII Delete Sum (khó)

Như bài này nhưng mỗi lần xóa ký tự `c` tốn `ord(c)` (mã ASCII), tính **tổng chi phí nhỏ nhất** để hai chuỗi giống nhau. Ví dụ `"sea"`, `"eat"` -> xóa `'s'` (115) và `'t'` (116) = `231`.

```python
exercise_5_ascii_sum("sea", "eat")      -> 231
exercise_5_ascii_sum("delete", "leet")  -> 403
exercise_5_ascii_sum("a", "a")          -> 0
```
Gợi ý: giữ nguyên khung DP, nhưng biên và phép cộng 1 thay bằng giá trị ký tự.

In [37]:
def exercise_5_ascii_sum(s1: str, s2: str) -> int:
    # TODO
    pass


# assert exercise_5_ascii_sum("sea", "eat") == 231
# assert exercise_5_ascii_sum("delete", "leet") == 403
# assert exercise_5_ascii_sum("a", "a") == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: hai chuỗi giống hệt thì cần bao nhiêu bước?
- Mức 2: `"abc"` và `"def"` không có ký tự chung, còn `"aaa"` là dãy con của `"aaaaa"`.
- Mức 3: kết quả `0`, `6`, `2`.

**Bài 2**
- Mức 1: hàng 0 là `[0, 1, 2]`, cột 0 là `0, 1, 2`.
- Mức 2: `f[1][1]`: `'a' != 'b'` -> `min(f[0][1], f[1][0]) + 1 = 2`. `f[1][2]`: `'a' == 'a'` -> `f[0][1] = 1`.
- Mức 3: bảng `[[0,1,2],[1,2,1],[2,1,2]]`, đáp án `2`.

**Bài 3**
- Mức 1: hai vòng `for i in range(1, m + 1)` ... đặt biên bằng hai vòng riêng.
- Mức 2: khi bằng nhau `f[i][j] = f[i-1][j-1]`.
- Mức 3: khi khác `f[i][j] = min(f[i-1][j], f[i][j-1]) + 1`; cuối cùng `return f[m][n]`.

**Bài 4**
- Mức 1: mảng `dp` dài `n + 1`, mọi phần tử bằng 0.
- Mức 2: dùng biến `prev` lưu ô chéo cũ trước khi ghi đè `dp[j]`.
- Mức 3: `return len(word1) + len(word2) - 2 * dp[-1]`.

**Bài 5**
- Mức 1: biên: `f[i][0] = f[i-1][0] + ord(s1[i-1])`, tương tự cột.
- Mức 2: khi bằng nhau giữ nguyên chéo; khi khác thì `min(f[i-1][j] + ord(s1[i-1]), f[i][j-1] + ord(s2[j-1]))`.
- Mức 3: trả `f[m][n]`; thử với oracle bitmask để chắc chắn.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [38]:
print(min_distance_2d("abc", "abc"))
print(min_distance_2d("abc", "def"))
print(min_distance_2d("aaa", "aaaaa"))

assert min_distance_2d("abc", "abc") == 0
assert min_distance_2d("abc", "def") == 6
assert min_distance_2d("aaa", "aaaaa") == 2

0
6
2


### Đáp án Bài 2

Bảng (hàng: `"", a, ab`; cột: `"", b, ba`):

| | "" | b | ba |
|---|---|---|---|
| "" | 0 | 1 | 2 |
| a | 1 | 2 | 1 |
| ab | 2 | 1 | 2 |

Đáp án `2`: giữ một ký tự (`a` hoặc `b`), xóa ký tự còn lại ở mỗi chuỗi.

In [39]:
hand_table_solution = [[0, 1, 2], [1, 2, 1], [2, 1, 2]]
hand_answer_solution = 2

computed = [[0] * 3 for _ in range(3)]
w1, w2 = "ab", "ba"
for i in range(3):
    computed[i][0] = i
for j in range(3):
    computed[0][j] = j
for i in range(1, 3):
    for j in range(1, 3):
        computed[i][j] = computed[i-1][j-1] if w1[i-1] == w2[j-1] else min(computed[i-1][j], computed[i][j-1]) + 1
assert computed == hand_table_solution
assert computed[2][2] == hand_answer_solution == min_distance_2d("ab", "ba")
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [40]:
def exercise_3_dp_solution(word1: str, word2: str) -> int:
    m, n = len(word1), len(word2)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        f[i][0] = i
    for j in range(1, n + 1):
        f[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                f[i][j] = f[i - 1][j - 1]
            else:
                f[i][j] = min(f[i - 1][j], f[i][j - 1]) + 1
    return f[m][n]

assert exercise_3_dp_solution("sea", "eat") == 2
assert exercise_3_dp_solution("leetcode", "etco") == 4
assert exercise_3_dp_solution("a", "b") == 2
random.seed(3)
for _ in range(100):
    s = "".join(random.choice("abc") for _ in range(random.randint(1, 7)))
    t = "".join(random.choice("abc") for _ in range(random.randint(1, 7)))
    assert exercise_3_dp_solution(s, t) == oracle(s, t)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [41]:
def exercise_4_lcs_form_solution(word1: str, word2: str) -> int:
    dp = [0] * (len(word2) + 1)
    for ch in word1:
        prev = 0
        for j in range(1, len(word2) + 1):
            tmp = dp[j]
            if ch == word2[j - 1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = tmp
    return len(word1) + len(word2) - 2 * dp[-1]

assert exercise_4_lcs_form_solution("sea", "eat") == 2
assert exercise_4_lcs_form_solution("leetcode", "etco") == 4
assert exercise_4_lcs_form_solution("ab", "ba") == 2
random.seed(4)
for _ in range(100):
    s = "".join(random.choice("abc") for _ in range(random.randint(1, 8)))
    t = "".join(random.choice("abc") for _ in range(random.randint(1, 8)))
    assert exercise_4_lcs_form_solution(s, t) == oracle(s, t)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [42]:
def exercise_5_ascii_sum_solution(s1: str, s2: str) -> int:
    m, n = len(s1), len(s2)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        f[i][0] = f[i - 1][0] + ord(s1[i - 1])
    for j in range(1, n + 1):
        f[0][j] = f[0][j - 1] + ord(s2[j - 1])
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                f[i][j] = f[i - 1][j - 1]
            else:
                f[i][j] = min(f[i - 1][j] + ord(s1[i - 1]), f[i][j - 1] + ord(s2[j - 1]))
    return f[m][n]

def ascii_oracle(s1: str, s2: str) -> int:
    best_keep = 0
    for mask in range(1 << len(s1)):
        sub = "".join(s1[k] for k in range(len(s1)) if mask >> k & 1)
        if is_subsequence(sub, s2):
            best_keep = max(best_keep, sum(map(ord, sub)))
    return sum(map(ord, s1)) + sum(map(ord, s2)) - 2 * best_keep

assert exercise_5_ascii_sum_solution("sea", "eat") == 231
assert exercise_5_ascii_sum_solution("delete", "leet") == 403
assert exercise_5_ascii_sum_solution("a", "a") == 0
random.seed(5)
for _ in range(150):
    s = "".join(random.choice("abcd") for _ in range(random.randint(1, 8)))
    t = "".join(random.choice("abcd") for _ in range(random.randint(1, 8)))
    assert exercise_5_ascii_sum_solution(s, t) == ascii_oracle(s, t), (s, t)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (bảng DP theo tiền tố)

```python
class Solution:
    def minDistance(self, word1: str, word2: str) -> int:
        m, n = len(word1), len(word2)
        f = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            f[i][0] = i                   # xóa hết i ký tự
        for j in range(1, n + 1):
            f[0][j] = j
        for i, a in enumerate(word1, 1):
            for j, b in enumerate(word2, 1):
                if a == b:
                    f[i][j] = f[i - 1][j - 1]
                else:
                    f[i][j] = min(f[i - 1][j], f[i][j - 1]) + 1
        return f[m][n]
```

Công thức nhanh: `đáp án = m + n - 2 * LCS(word1, word2)`.

### Checklist trước khi submit

- [ ] Hàng/cột 0 là biên và được đặt `i`, `j` (không phải 0)?
- [ ] Ký tự thật là `word[i - 1]`, không phải `word[i]`?
- [ ] Bảng tạo bằng list comprehension, không phải `[[0]*n]*m`?
- [ ] Khi khác nhau chỉ lấy `min(trên, trái) + 1` (không có đường chéo)?
- [ ] Nếu dùng LCS: có `m + n - 2 * LCS` (nhân đôi)?
- [ ] Nếu đệ quy có nhớ: đã nâng `sys.setrecursionlimit` (độ sâu tới 1000)?
- [ ] Đã thử `("a","b")`, `("ab","ba")`, hai chuỗi bằng nhau, một chuỗi là dãy con của chuỗi kia?
- [ ] Tên method đúng `minDistance(self, word1, word2)` và trả `int`?